<div style="border:1px solid #d9e1ea;border-left:6px solid #2d8a57;border-radius:14px;padding:18px 20px;background:#fff;"><h1 style="margin:0 0 6px;color:#16213b;">Pydantic — Complete Learning Notebook</h1><p style="margin:0;color:#63738a;">BaseModel, fields, types, nested models, Literal, validation and serialization — mapped to app/analytics/models/request.py and app/main.py.</p><p style="margin:10px 0 0;"><code>notebooks/pydantic/pydantic_project_learning.ipynb</code></p></div>

![Contract](assets/01_contract.svg)

## 1. What Pydantic Is

Pydantic is a Python library for defining, validating, and parsing structured data using type hints.

## 2. BaseModel

In [7]:
from pydantic import BaseModel, Field

class AskRequest(BaseModel):
    question: str

## 3. Optional Fields

In [30]:
#  business contracts are Pydantic
class ReservationQuery(BaseModel):  
    country: str | None = None
    product: str | None = None
    campaign_id: str | None = None

Missing fields stay `None`, which supports the project's **clarify instead of guess** rule.

## 4. Literal

In [19]:
from typing import Literal

class ExtractedRequest(BaseModel):
    intent: Literal["knowledge", "analytics"]
    metric: str | None = None
    query: ReservationQuery = Field(default_factory=ReservationQuery)

## 5. Field(default_factory=...)

In [20]:
query: ReservationQuery = Field(default_factory=ReservationQuery)

query

FieldInfo(annotation=NoneType, required=False, default_factory=ReservationQuery)

## 6. Nested Models

In [21]:
# Although query is defined as ReservationQuery,
# we can pass a normal Python dictionary here.
# Pydantic automatically validates and converts
# this dictionary into a ReservationQuery object.

request = ExtractedRequest(
    intent="analytics",
    metric="reserved_users",
    query={"country": "Germany", "product": "Phone Mi 17 Pro", "campaign_id": "CMP001"},
)
request

ExtractedRequest(intent='analytics', metric='reserved_users', query=ReservationQuery(country='Germany', product='Phone Mi 17 Pro', campaign_id='CMP001'))

## 7. Validation Errors

In [22]:
from pydantic import ValidationError

try:
    ExtractedRequest(intent="other")
except ValidationError as exc:
    print(exc)

1 validation error for ExtractedRequest
intent
  Input should be 'knowledge' or 'analytics' [type=literal_error, input_value='other', input_type=str]
    For further information visit https://errors.pydantic.dev/2.13/v/literal_error


## 8. model_dump() / model_validate()

In [26]:
payload = request.model_dump()
again = ExtractedRequest.model_validate(payload)
print(payload)
again

{'intent': 'analytics', 'metric': 'reserved_users', 'query': {'country': 'Germany', 'product': 'Phone Mi 17 Pro', 'campaign_id': 'CMP001'}}


ExtractedRequest(intent='analytics', metric='reserved_users', query=ReservationQuery(country='Germany', product='Phone Mi 17 Pro', campaign_id='CMP001'))

## 9. Custom Validators

In [28]:
from pydantic import field_validator

@field_validator("campaign_month")
@classmethod
def valid_month(cls, value):
    if value is not None and not 1 <= value <= 12:
        raise ValueError("campaign_month must be 1..12")
    return value

## 10. Pydantic vs TypedDict

![Models](assets/02_models.svg)

`BaseModel` performs runtime validation/parsing. 

`TypedDict` mainly describes dict shape for typing. 

In the project, business contracts are Pydantic while `AgentState` is TypedDict.

## 11. Pydantic + LLM Structured Output

In [ ]:
llm = ChatOpenAI(...).with_structured_output(ExtractedRequest)

## Q&A — Fast Review

<details open><summary><b>Q1. Why Pydantic with LLMs?</b></summary>

**Answer:** It converts **model output into a typed contract that can be validated** before downstream execution.

</details>

<details open><summary><b>Q2. BaseModel vs TypedDict?</b></summary>

**Answer:** BaseModel validates at runtime; TypedDict mainly provides type hints.

</details>

<details open><summary><b>Q3. Why Literal for intent?</b></summary>

**Answer:** It constrains routing to known values.

</details>

<details open><summary><b>Q4. Why keep missing fields None?</b></summary>

**Answer:** To clarify missing context instead of guessing.

</details>

<details open><summary><b>Q5. What is model_dump()?</b></summary>

**Answer:** It serializes a validated model into a plain dict.

</details>

## Classic Architecture Q&A — Memorize This

### Q. Why do you use LangChain selectively instead of making it the entire application framework?

> **I use LangChain selectively rather than making it the entire application framework. LangChain's ChatOpenAI integration handles structured extraction, LangGraph handles stateful workflow orchestration, and LlamaIndex with FAISS handles the knowledge RAG layer. This keeps responsibilities explicit and prevents the LLM from directly controlling analytics SQL.**

<div style="background:#eef7ff;border:1px solid #c9e0f2;border-radius:10px;padding:10px 12px;margin:10px 0;">
</div>

### Memory Map

```text
LangChain / ChatOpenAI  → Structured Extraction
Pydantic                → Typed Contract
LangGraph               → Stateful Workflow
LlamaIndex + FAISS      → Knowledge RAG
Controlled SQL          → Trusted Numbers
FastAPI                 → Service API
```

### One-line takeaway

> **Do not force every responsibility into one framework. Keep the boundaries explicit.**